# CS541 Course Project: ErrorLock

This notebook implements the complete A--F experiment plan:

A. CASTLE baseline for each model  
B. CASTLE split-based ErrorLock: build ErrorLock from CASTLE train mistakes, test on held-out CASTLE  
C. CASTLE same-dataset ErrorLock: build ErrorLock from all CASTLE mistakes, test on all CASTLE  
D. Build Juliet external benchmark subset with overlapping CWEs  
E. Juliet baseline for each model  
F. Juliet + CASTLE-built ErrorLock: test Juliet using ErrorLock database built only from CASTLE mistakes  

Note: Experiment C is in-dataset memory analysis, not the main generalization result. Experiments B and F are more defensible.

## 0. Install packages

In [1]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

!nvidia-smi

torch: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
Device: Tesla T4
Fri May  8 16:35:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                    

In [2]:
!pip install -q -U \
  "transformers>=4.46,<4.57" \
  "accelerate>=1.1,<2.0" \
  "datasets>=2.21,<4.0" \
  "bitsandbytes>=0.46.0" \
  "sentencepiece" \
  "jedi>=0.16"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.7 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np
import pandas as pd
import sklearn
import datasets
import transformers
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)

!nvidia-smi

torch: 2.10.0+cu128
CUDA available: True
Device: Tesla T4
numpy: 2.0.2
pandas: 2.2.2
sklearn: 1.6.1
datasets: 3.6.0
transformers: 4.56.2
bitsandbytes: 0.49.2
Fri May  8 16:36:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       3MiB /  15360MiB |      

## 1. Imports and global settings

In [4]:
import os
import gc
import re
import json
import time
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

warnings.filterwarnings("ignore")

RESULT_DIR = Path("errorlock_results")
DB_DIR = Path("errorlock_dbs")
DATA_DIR = Path("errorlock_data")

RESULT_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
JULIET_PAIR_COUNT = 125       # 125 good + 125 bad = 250 Juliet samples
TOP_K_MISTAKES = 3
MAX_NEW_TOKENS = 256

## 2. Model configuration

In [5]:
MODEL_CONFIGS = [
    {
        "short_name": "qwen25_coder_7b",
        "model_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
        "max_length": 4096,
    },
    {
        "short_name": "codellama_7b_instruct",
        "model_id": "codellama/CodeLlama-7b-Instruct-hf",
        "max_length": 4096,
    },
    # Optional smaller model. Uncomment if you have time.
    # {
    #     "short_name": "starcoder2_3b",
    #     "model_id": "bigcode/starcoder2-3b",
    #     "max_length": 4096,
    # },
]

# Safe first run: only Qwen.
RUN_MODEL_CONFIGS = [MODEL_CONFIGS[0]]

# After Qwen works, use this instead:
# RUN_MODEL_CONFIGS = MODEL_CONFIGS

## 3. Load CASTLE-C250

In [6]:
CASTLE_URL = "https://raw.githubusercontent.com/CASTLE-Benchmark/CASTLE-Benchmark/main/datasets/CASTLE-C250.min.json"
CASTLE_PATH = DATA_DIR / "CASTLE-C250.min.json"

if not CASTLE_PATH.exists():
    !wget -q -O {CASTLE_PATH} {CASTLE_URL}

with open(CASTLE_PATH, "r") as f:
    castle_obj = json.load(f)

castle_df = pd.json_normalize(castle_obj["tests"])
dataset_prompt = castle_obj.get("prompt", "")

castle_df["id"] = castle_df["id"].astype(str)
castle_df["code"] = castle_df["code"].fillna("")
castle_df["label"] = castle_df["vulnerable"].astype(int)
castle_df["vulnerable"] = castle_df["label"]
castle_df["cwe"] = castle_df["cwe"].astype(int)
castle_df["source"] = "castle"

if "lines" not in castle_df.columns:
    castle_df["lines"] = [[] for _ in range(len(castle_df))]

print("CASTLE shape:", castle_df.shape)
print("CASTLE label counts:")
print(castle_df["label"].value_counts())
print("CASTLE CWE count:", castle_df["cwe"].nunique())
print("CASTLE CWEs:", sorted(castle_df["cwe"].unique()))

CASTLE shape: (250, 21)
CASTLE label counts:
label
1    150
0    100
Name: count, dtype: int64
CASTLE CWE count: 25
CASTLE CWEs: [np.int64(22), np.int64(78), np.int64(89), np.int64(125), np.int64(134), np.int64(190), np.int64(253), np.int64(327), np.int64(362), np.int64(369), np.int64(401), np.int64(415), np.int64(416), np.int64(476), np.int64(522), np.int64(617), np.int64(628), np.int64(674), np.int64(761), np.int64(770), np.int64(787), np.int64(798), np.int64(822), np.int64(835), np.int64(843)]


## 4. CASTLE train/test split for Experiment B

In [7]:
castle_df["strat_key"] = castle_df["cwe"].astype(str) + "_" + castle_df["label"].astype(str)

castle_train_df, castle_test_df = train_test_split(
    castle_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=castle_df["strat_key"]
)

castle_train_ids = set(castle_train_df["id"].astype(str))
castle_test_ids = set(castle_test_df["id"].astype(str))

print("CASTLE train:", castle_train_df.shape)
print("CASTLE test:", castle_test_df.shape)
print("Train label counts:")
print(castle_train_df["label"].value_counts())
print("Test label counts:")
print(castle_test_df["label"].value_counts())

CASTLE train: (200, 22)
CASTLE test: (50, 22)
Train label counts:
label
1    125
0     75
Name: count, dtype: int64
Test label counts:
label
1    25
0    25
Name: count, dtype: int64


## 5. Load Juliet and build external CWE-overlap benchmark

In [8]:
juliet = load_dataset("LorenzH/juliet_test_suite_c_1_3")
juliet_test = juliet["test"].to_pandas()

print(juliet)
print("Juliet test shape:", juliet_test.shape)
print("Juliet columns:", list(juliet_test.columns))
juliet_test.head()

README.md: 0.00B [00:00, ?B/s]

jts_c_1_3_train.csv:   0%|          | 0.00/237M [00:00<?, ?B/s]

jts_c_1_3_test.csv:   0%|          | 0.00/59.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/80706 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20177 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['index', 'filename', 'class', 'good', 'bad'],
        num_rows: 80706
    })
    test: Dataset({
        features: ['index', 'filename', 'class', 'good', 'bad'],
        num_rows: 20177
    })
})
Juliet test shape: (20177, 5)
Juliet columns: ['index', 'filename', 'class', 'good', 'bad']


,index,filename,class,good,bad
0,3,testcases/CWE591_Sensitive_Data_Storage_in_Imp...,111,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...
1,4,testcases/CWE591_Sensitive_Data_Storage_in_Imp...,111,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...
2,14,testcases/CWE591_Sensitive_Data_Storage_in_Imp...,111,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...
3,17,testcases/CWE591_Sensitive_Data_Storage_in_Imp...,111,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...
4,23,testcases/CWE591_Sensitive_Data_Storage_in_Imp...,111,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...


In [9]:
def build_juliet_eval_df(juliet_df, castle_cwes, max_pairs=125, random_state=42):
    tmp = juliet_df.copy()
    tmp["class"] = tmp["class"].astype(int)

    overlap = tmp[tmp["class"].isin(castle_cwes)].copy()

    print("Full Juliet rows:", len(tmp))
    print("Overlap Juliet rows:", len(overlap))
    print("Overlapping CWEs:", sorted(overlap["class"].unique()))

    if len(overlap) == 0:
        print("Warning: no CWE overlap found. Sampling from all Juliet instead.")
        overlap = tmp

    sampled = overlap.sample(
        n=min(max_pairs, len(overlap)),
        random_state=random_state
    )

    rows = []

    for _, row in sampled.iterrows():
        cwe = int(row["class"])
        filename = str(row["filename"])

        good_code = "" if pd.isna(row["good"]) else str(row["good"])
        bad_code = "" if pd.isna(row["bad"]) else str(row["bad"])

        rows.append({
            "id": f"juliet::{filename}::good",
            "code": good_code,
            "vulnerable": 0,
            "label": 0,
            "cwe": cwe,
            "lines": [],
            "source": "juliet",
        })

        rows.append({
            "id": f"juliet::{filename}::bad",
            "code": bad_code,
            "vulnerable": 1,
            "label": 1,
            "cwe": cwe,
            "lines": [],
            "source": "juliet",
        })

    out = pd.DataFrame(rows)
    out["code"] = out["code"].fillna("")
    out["cwe"] = out["cwe"].astype(int)
    out["vulnerable"] = out["vulnerable"].astype(int)
    out["label"] = out["label"].astype(int)

    return out

In [10]:
castle_cwes = set(castle_df["cwe"].astype(int).unique())

juliet_eval_df = build_juliet_eval_df(
    juliet_test,
    castle_cwes=castle_cwes,
    max_pairs=JULIET_PAIR_COUNT,
    random_state=RANDOM_STATE
)

juliet_path = RESULT_DIR / "juliet_overlap_250.csv"
juliet_eval_df.to_csv(juliet_path, index=False)

print("Juliet eval shape:", juliet_eval_df.shape)
print("Juliet label counts:")
print(juliet_eval_df["label"].value_counts())
print("Juliet CWE count:", juliet_eval_df["cwe"].nunique())
print("Saved to:", juliet_path)
juliet_eval_df.head()

Full Juliet rows: 20177
Overlap Juliet rows: 964
Overlapping CWEs: [np.int64(22), np.int64(78), np.int64(89), np.int64(125), np.int64(134), np.int64(190)]
Juliet eval shape: (250, 7)
Juliet label counts:
label
0    125
1    125
Name: count, dtype: int64
Juliet CWE count: 6
Saved to: errorlock_results/juliet_overlap_250.csv


,id,code,vulnerable,label,cwe,lines,source
0,juliet::testcases/CWE122_Heap_Based_Buffer_Ove...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,0,0,134,[],juliet
1,juliet::testcases/CWE122_Heap_Based_Buffer_Ove...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,1,1,134,[],juliet
2,juliet::testcases/CWE122_Heap_Based_Buffer_Ove...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,0,0,125,[],juliet
3,juliet::testcases/CWE122_Heap_Based_Buffer_Ove...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,1,1,125,[],juliet
4,juliet::testcases/CWE122_Heap_Based_Buffer_Ove...,/* TEMPLATE GENERATED TESTCASE FILE\nFilename:...,0,0,125,[],juliet


## 6. Model and generation utility functions

In [11]:
def safe_json_dumps(x):
    try:
        return json.dumps(x)
    except TypeError:
        return json.dumps(str(x))


def unload_model():
    global model, tokenizer
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_llm(model_id):
    print("Loading model:", model_id)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        print("CUDA is available. Loading model with 4-bit bitsandbytes quantization.")

        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.float16,
            quantization_config=quantization_config,
            trust_remote_code=True
        )
    else:
        print("CUDA is NOT available. Loading without bitsandbytes quantization.")
        print("Warning: 7B models may be too slow or too large on CPU.")

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.float32,
            trust_remote_code=True
        )

    model.eval()
    return tokenizer, model

In [12]:
def build_base_prompt(code: str) -> str:
    return f"""{dataset_prompt}

C code:
```c
{code}
```"""


def apply_prompt_template(tokenizer, user_prompt: str):
    messages = [{"role": "user", "content": user_prompt}]

    try:
        if getattr(tokenizer, "chat_template", None):
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
    except Exception:
        pass

    return f"Instruction:\n{user_prompt}\n\nAnswer:\n"


def generate_text(tokenizer, model, user_prompt: str, max_new_tokens: int = 256, max_length: int = 4096) -> str:
    prompt_text = apply_prompt_template(tokenizer, user_prompt)

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    prompt_decoded = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)

    if decoded.startswith(prompt_decoded):
        return decoded[len(prompt_decoded):].strip()

    return decoded.strip()

## 7. Output parser

In [13]:
def extract_json_list_block(text: str):
    if text is None:
        return None

    m = re.search(r"```json\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    m = re.search(r"```\s*(.*?)\s*```", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    start = text.find("[")
    end = text.rfind("]")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            pass

    return None


def pred_from_castle_response(raw: str):
    vuln_list = extract_json_list_block(raw)

    if not isinstance(vuln_list, list):
        return None, [], None

    pred_label = int(len(vuln_list) > 0)
    pred_lines = []
    pred_cwe = None

    for item in vuln_list:
        if not isinstance(item, dict):
            continue

        try:
            if "line" in item and item["line"] is not None:
                pred_lines.append(int(item["line"]))
        except Exception:
            pass

        if pred_cwe is None and "cwe" in item:
            try:
                c = int(item["cwe"])
                if c != 0:
                    pred_cwe = c
            except Exception:
                pass

    pred_lines = sorted(set(pred_lines))
    return pred_label, pred_lines, pred_cwe

## 8. Baseline runner

In [14]:
def run_baseline_prediction(
    eval_df,
    tokenizer,
    model,
    model_short_name,
    dataset_name,
    save_path,
    max_new_tokens=256,
    max_length=4096,
    save_every=10,
    resume=True
):
    save_path = Path(save_path)
    done_ids = set()
    rows = []

    if resume and save_path.exists():
        old_df = pd.read_csv(save_path)
        rows = old_df.to_dict("records")
        done_ids = set(old_df["id"].astype(str))
        print(f"Resuming from {save_path}. Already done: {len(done_ids)}")

    start = time.time()

    for r in tqdm(eval_df.itertuples(index=False), total=len(eval_df)):
        sample_id = str(getattr(r, "id"))

        if sample_id in done_ids:
            continue

        code = getattr(r, "code")
        prompt = build_base_prompt(code)

        raw = generate_text(
            tokenizer,
            model,
            prompt,
            max_new_tokens=max_new_tokens,
            max_length=max_length
        )

        pred_label, pred_lines, pred_cwe = pred_from_castle_response(raw)

        rows.append({
            "model": model_short_name,
            "dataset": dataset_name,
            "method": "baseline",
            "id": sample_id,
            "true_label": int(getattr(r, "vulnerable")),
            "true_cwe": int(getattr(r, "cwe")),
            "true_lines": safe_json_dumps(getattr(r, "lines")),
            "pred_label": pred_label,
            "pred_cwe": pred_cwe,
            "pred_lines": safe_json_dumps(pred_lines),
            "raw": raw,
        })

        if len(rows) % save_every == 0:
            pd.DataFrame(rows).to_csv(save_path, index=False)

    out_df = pd.DataFrame(rows)
    out_df.to_csv(save_path, index=False)

    print(f"Saved baseline predictions to {save_path}")
    print(f"Runtime: {(time.time() - start) / 60:.2f} minutes")
    return out_df

## 9. Evaluation helpers

In [15]:
def evaluate_prediction_df(pred_df, model_name, dataset_name, method_name):
    total = len(pred_df)
    valid = pred_df.dropna(subset=["pred_label"]).copy()

    if len(valid) == 0:
        return {
            "model": model_name,
            "dataset": dataset_name,
            "method": method_name,
            "total": total,
            "parsed": 0,
            "parsed_rate": 0.0,
            "accuracy": None,
            "precision_macro": None,
            "recall_macro": None,
            "f1_macro": None,
            "precision_weighted": None,
            "recall_weighted": None,
            "f1_weighted": None,
            "tn": None,
            "fp": None,
            "fn": None,
            "tp": None,
        }

    valid["true_label"] = valid["true_label"].astype(int)
    valid["pred_label"] = valid["pred_label"].astype(int)

    acc = accuracy_score(valid["true_label"], valid["pred_label"])

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        valid["true_label"],
        valid["pred_label"],
        average="macro",
        zero_division=0
    )

    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        valid["true_label"],
        valid["pred_label"],
        average="weighted",
        zero_division=0
    )

    cm = confusion_matrix(valid["true_label"], valid["pred_label"], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "model": model_name,
        "dataset": dataset_name,
        "method": method_name,
        "total": total,
        "parsed": len(valid),
        "parsed_rate": len(valid) / total,
        "accuracy": acc,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "precision_weighted": p_weighted,
        "recall_weighted": r_weighted,
        "f1_weighted": f1_weighted,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def print_detailed_report(pred_df):
    valid = pred_df.dropna(subset=["pred_label"]).copy()
    valid["true_label"] = valid["true_label"].astype(int)
    valid["pred_label"] = valid["pred_label"].astype(int)

    print("Parsed:", len(valid), "/", len(pred_df))
    print(confusion_matrix(valid["true_label"], valid["pred_label"], labels=[0, 1]))
    print(classification_report(valid["true_label"], valid["pred_label"], digits=4, zero_division=0))

## 10. ErrorLock database and retrieval

In [16]:
def error_type_from_row(row):
    if int(row["true_label"]) == int(row["pred_label"]):
        return "correct"

    if int(row["true_label"]) == 0 and int(row["pred_label"]) == 1:
        return "false_positive"

    if int(row["true_label"]) == 1 and int(row["pred_label"]) == 0:
        return "false_negative"

    return "wrong_prediction"


def build_errorlock_database(
    baseline_pred_df,
    source_code_df,
    allowed_ids,
    db_path,
    model_short_name,
    source_name
):
    db_path = Path(db_path)
    allowed_ids = set(str(x) for x in allowed_ids)

    pred = baseline_pred_df.copy()
    pred["id"] = pred["id"].astype(str)
    pred = pred[pred["id"].isin(allowed_ids)]
    pred = pred.dropna(subset=["pred_label"]).copy()

    source_cols = source_code_df[["id", "code"]].copy()
    source_cols["id"] = source_cols["id"].astype(str)

    full = pred.merge(source_cols, on="id", how="left")
    full["correct"] = full["true_label"].astype(int) == full["pred_label"].astype(int)
    full["error_type"] = full.apply(error_type_from_row, axis=1)

    mistakes = full[full["correct"] == False].copy()

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS mistakes")

    cur.execute("""
    CREATE TABLE mistakes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        model_name TEXT,
        source_name TEXT,
        sample_id TEXT,
        true_label INTEGER,
        true_cwe INTEGER,
        pred_label INTEGER,
        pred_cwe INTEGER,
        true_lines TEXT,
        pred_lines TEXT,
        error_type TEXT,
        code TEXT,
        raw_output TEXT
    )
    """)

    for _, row in mistakes.iterrows():
        cur.execute("""
        INSERT INTO mistakes (
            model_name, source_name, sample_id,
            true_label, true_cwe, pred_label, pred_cwe,
            true_lines, pred_lines, error_type, code, raw_output
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            model_short_name,
            source_name,
            str(row["id"]),
            int(row["true_label"]),
            int(row["true_cwe"]) if pd.notna(row["true_cwe"]) else None,
            int(row["pred_label"]) if pd.notna(row["pred_label"]) else None,
            int(row["pred_cwe"]) if pd.notna(row["pred_cwe"]) else None,
            str(row["true_lines"]),
            str(row["pred_lines"]),
            row["error_type"],
            row["code"],
            row["raw"],
        ))

    conn.commit()
    conn.close()

    print(f"Built ErrorLock DB: {db_path}")
    print("Mistakes stored:", len(mistakes))
    if len(mistakes) > 0:
        print(mistakes["error_type"].value_counts())

    return mistakes

In [17]:
class ErrorLockRetriever:
    def __init__(self, db_path):
        self.db_path = Path(db_path)
        self.records = self._load_records()

        if len(self.records) == 0:
            self.vectorizer = None
            self.matrix = None
            return

        self.records["retrieval_text"] = self.records.apply(self._make_mistake_text, axis=1)

        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b[\w_]+\b",
            max_features=8000
        )

        self.matrix = self.vectorizer.fit_transform(self.records["retrieval_text"])

    def _load_records(self):
        conn = sqlite3.connect(self.db_path)
        records = pd.read_sql_query("SELECT * FROM mistakes", conn)
        conn.close()
        return records

    def _make_mistake_text(self, row):
        return f"""
sample_id: {row['sample_id']}
error_type: {row['error_type']}
true_label: {row['true_label']}
true_cwe: CWE-{row['true_cwe']}
pred_label: {row['pred_label']}
pred_cwe: CWE-{row['pred_cwe']}
code:
{row['code']}
"""

    def retrieve(self, code, true_cwe=None, top_k=3):
        if self.vectorizer is None or self.matrix is None or len(self.records) == 0:
            return []

        query = code
        if true_cwe is not None:
            query = f"CWE-{true_cwe}\n{code}"

        q = self.vectorizer.transform([query])
        sims = cosine_similarity(q, self.matrix)[0]
        top_indices = sims.argsort()[::-1][:top_k]

        results = []

        for idx in top_indices:
            row = self.records.iloc[idx]

            results.append({
                "sample_id": row["sample_id"],
                "score": float(sims[idx]),
                "error_type": row["error_type"],
                "true_label": row["true_label"],
                "true_cwe": row["true_cwe"],
                "pred_label": row["pred_label"],
                "pred_cwe": row["pred_cwe"],
                "code": row["code"],
                "raw_output": row["raw_output"],
            })

        return results

## 11. ErrorLock prompt and runner

In [18]:
def format_mistakes_for_prompt(mistakes):
    if len(mistakes) == 0:
        return "No similar past mistakes were found."

    parts = []

    for i, m in enumerate(mistakes, start=1):
        if m["error_type"] == "false_positive":
            lesson = (
                "In this past case, the model incorrectly reported a vulnerability in safe code. "
                "Be careful not to over-report similar code patterns."
            )
        elif m["error_type"] == "false_negative":
            lesson = (
                "In this past case, the model missed a real vulnerability. "
                "Be careful not to ignore similar risky patterns."
            )
        else:
            lesson = (
                "In this past case, the model made an incorrect prediction. "
                "Compare the current code carefully against this mistake."
            )

        parts.append(f"""
Past mistake {i}:
Similarity score: {m['score']:.4f}
Error type: {m['error_type']}
Ground truth label: {m['true_label']}
Ground truth CWE: {m['true_cwe']}
Model predicted label: {m['pred_label']}
Model predicted CWE: {m['pred_cwe']}
Lesson: {lesson}
""")

    return "\n".join(parts)


def build_errorlock_prompt(code: str, retrieved_mistakes) -> str:
    mistake_text = format_mistakes_for_prompt(retrieved_mistakes)

    return f"""{dataset_prompt}

Before analyzing the new code, review these similar past mistakes made by the model.
Use them only as guidance. Do not blindly copy their labels.
The goal is to avoid repeating similar false positives or false negatives.

{mistake_text}

Now analyze the following C code.
Return the answer using the exact required JSON format.

C code:
```c
{code}
```"""

In [19]:
def run_errorlock_prediction(
    eval_df,
    tokenizer,
    model,
    retriever,
    model_short_name,
    dataset_name,
    method_name,
    save_path,
    top_k=3,
    max_new_tokens=256,
    max_length=4096,
    save_every=10,
    resume=True
):
    save_path = Path(save_path)
    done_ids = set()
    rows = []

    if resume and save_path.exists():
        old_df = pd.read_csv(save_path)
        rows = old_df.to_dict("records")
        done_ids = set(old_df["id"].astype(str))
        print(f"Resuming from {save_path}. Already done: {len(done_ids)}")

    start = time.time()

    for r in tqdm(eval_df.itertuples(index=False), total=len(eval_df)):
        sample_id = str(getattr(r, "id"))

        if sample_id in done_ids:
            continue

        code = getattr(r, "code")
        true_cwe = int(getattr(r, "cwe"))

        retrieved = retriever.retrieve(code, true_cwe=true_cwe, top_k=top_k)
        prompt = build_errorlock_prompt(code, retrieved)

        raw = generate_text(
            tokenizer,
            model,
            prompt,
            max_new_tokens=max_new_tokens,
            max_length=max_length
        )

        pred_label, pred_lines, pred_cwe = pred_from_castle_response(raw)

        rows.append({
            "model": model_short_name,
            "dataset": dataset_name,
            "method": method_name,
            "id": sample_id,
            "true_label": int(getattr(r, "vulnerable")),
            "true_cwe": true_cwe,
            "true_lines": safe_json_dumps(getattr(r, "lines")),
            "pred_label": pred_label,
            "pred_cwe": pred_cwe,
            "pred_lines": safe_json_dumps(pred_lines),

            "retrieved_mistake_ids": safe_json_dumps([m["sample_id"] for m in retrieved]),
            "retrieved_scores": safe_json_dumps([m["score"] for m in retrieved]),
            "retrieved_error_types": safe_json_dumps([m["error_type"] for m in retrieved]),
            "retrieved_true_cwes": safe_json_dumps([m["true_cwe"] for m in retrieved]),
            "retrieved_pred_cwes": safe_json_dumps([m["pred_cwe"] for m in retrieved]),

            "raw": raw,
        })

        if len(rows) % save_every == 0:
            pd.DataFrame(rows).to_csv(save_path, index=False)

    out_df = pd.DataFrame(rows)
    out_df.to_csv(save_path, index=False)

    print(f"Saved ErrorLock predictions to {save_path}")
    print(f"Runtime: {(time.time() - start) / 60:.2f} minutes")

    return out_df

## 12. Main A--F experiment loop

In [20]:
all_metrics = []

for cfg in RUN_MODEL_CONFIGS:
    model_short = cfg["short_name"]
    model_id = cfg["model_id"]
    max_length = cfg.get("max_length", 4096)

    print("\n" + "=" * 80)
    print("Running model:", model_short)
    print("=" * 80)

    unload_model()
    tokenizer, model = load_llm(model_id)

    # A. CASTLE baseline on all 250 cases
    castle_all_baseline_path = RESULT_DIR / f"{model_short}__A_castle_all_baseline.csv"

    castle_all_baseline = run_baseline_prediction(
        eval_df=castle_df,
        tokenizer=tokenizer,
        model=model,
        model_short_name=model_short,
        dataset_name="CASTLE_all_250",
        save_path=castle_all_baseline_path,
        max_new_tokens=MAX_NEW_TOKENS,
        max_length=max_length,
        save_every=10,
        resume=True
    )

    all_metrics.append(evaluate_prediction_df(castle_all_baseline, model_short, "CASTLE_all_250", "A_baseline"))

    # B. CASTLE split-based ErrorLock
    castle_split_db_path = DB_DIR / f"{model_short}__B_castle_train_mistakes.db"

    build_errorlock_database(
        baseline_pred_df=castle_all_baseline,
        source_code_df=castle_df,
        allowed_ids=castle_train_ids,
        db_path=castle_split_db_path,
        model_short_name=model_short,
        source_name="CASTLE_train_split"
    )

    split_retriever = ErrorLockRetriever(castle_split_db_path)

    castle_test_baseline = castle_all_baseline[
        castle_all_baseline["id"].astype(str).isin(castle_test_ids)
    ].copy()

    all_metrics.append(evaluate_prediction_df(castle_test_baseline, model_short, "CASTLE_heldout_50", "B0_baseline_on_heldout"))

    castle_split_errorlock_path = RESULT_DIR / f"{model_short}__B_castle_heldout_errorlock.csv"

    castle_split_errorlock = run_errorlock_prediction(
        eval_df=castle_test_df,
        tokenizer=tokenizer,
        model=model,
        retriever=split_retriever,
        model_short_name=model_short,
        dataset_name="CASTLE_heldout_50",
        method_name="B_errorlock_train_split",
        save_path=castle_split_errorlock_path,
        top_k=TOP_K_MISTAKES,
        max_new_tokens=MAX_NEW_TOKENS,
        max_length=max_length,
        save_every=10,
        resume=True
    )

    all_metrics.append(evaluate_prediction_df(castle_split_errorlock, model_short, "CASTLE_heldout_50", "B_errorlock_train_split"))

    # C. CASTLE same-dataset ErrorLock
    castle_all_db_path = DB_DIR / f"{model_short}__C_castle_all_mistakes.db"

    build_errorlock_database(
        baseline_pred_df=castle_all_baseline,
        source_code_df=castle_df,
        allowed_ids=set(castle_df["id"].astype(str)),
        db_path=castle_all_db_path,
        model_short_name=model_short,
        source_name="CASTLE_all_250"
    )

    castle_all_retriever = ErrorLockRetriever(castle_all_db_path)

    castle_all_errorlock_path = RESULT_DIR / f"{model_short}__C_castle_all_errorlock.csv"

    castle_all_errorlock = run_errorlock_prediction(
        eval_df=castle_df,
        tokenizer=tokenizer,
        model=model,
        retriever=castle_all_retriever,
        model_short_name=model_short,
        dataset_name="CASTLE_all_250",
        method_name="C_errorlock_same_dataset",
        save_path=castle_all_errorlock_path,
        top_k=TOP_K_MISTAKES,
        max_new_tokens=MAX_NEW_TOKENS,
        max_length=max_length,
        save_every=10,
        resume=True
    )

    all_metrics.append(evaluate_prediction_df(castle_all_errorlock, model_short, "CASTLE_all_250", "C_errorlock_same_dataset"))

    # E. Juliet baseline
    juliet_baseline_path = RESULT_DIR / f"{model_short}__E_juliet_baseline.csv"

    juliet_baseline = run_baseline_prediction(
        eval_df=juliet_eval_df,
        tokenizer=tokenizer,
        model=model,
        model_short_name=model_short,
        dataset_name="Juliet_overlap_250",
        save_path=juliet_baseline_path,
        max_new_tokens=MAX_NEW_TOKENS,
        max_length=max_length,
        save_every=10,
        resume=True
    )

    all_metrics.append(evaluate_prediction_df(juliet_baseline, model_short, "Juliet_overlap_250", "E_baseline"))

    # F. Juliet + CASTLE-built ErrorLock
    juliet_errorlock_path = RESULT_DIR / f"{model_short}__F_juliet_castle_errorlock.csv"

    juliet_errorlock = run_errorlock_prediction(
        eval_df=juliet_eval_df,
        tokenizer=tokenizer,
        model=model,
        retriever=castle_all_retriever,
        model_short_name=model_short,
        dataset_name="Juliet_overlap_250",
        method_name="F_errorlock_from_castle",
        save_path=juliet_errorlock_path,
        top_k=TOP_K_MISTAKES,
        max_new_tokens=MAX_NEW_TOKENS,
        max_length=max_length,
        save_every=10,
        resume=True
    )

    all_metrics.append(evaluate_prediction_df(juliet_errorlock, model_short, "Juliet_overlap_250", "F_errorlock_from_castle"))

    metrics_df = pd.DataFrame(all_metrics)
    metrics_df.to_csv(RESULT_DIR / "summary_metrics_partial.csv", index=False)

    print("\nCurrent metrics:")
    display(metrics_df)

    unload_model()

metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(RESULT_DIR / "summary_metrics_final.csv", index=False)

metrics_df


Running model: qwen25_coder_7b
Loading model: Qwen/Qwen2.5-Coder-7B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Saved baseline predictions to errorlock_results/qwen25_coder_7b__E_juliet_baseline.csv
Runtime: 26.36 minutes


  0%|          | 0/250 [00:00<?, ?it/s]

Saved ErrorLock predictions to errorlock_results/qwen25_coder_7b__F_juliet_castle_errorlock.csv
Runtime: 22.41 minutes

Current metrics:


,model,dataset,method,total,parsed,parsed_rate,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,tn,fp,fn,tp
0,qwen25_coder_7b,CASTLE_all_250,A_baseline,250,237,0.948,0.649789,0.624871,0.608508,0.609648,0.638211,0.649789,0.637641,39,53,30,115
1,qwen25_coder_7b,CASTLE_heldout_50,B0_baseline_on_heldout,50,47,0.940,0.680851,0.686275,0.672727,0.671329,0.685023,0.680851,0.674900,12,10,5,20
2,qwen25_coder_7b,CASTLE_heldout_50,B_errorlock_train_split,50,50,1.000,0.580000,0.581169,0.580000,0.578483,0.581169,0.580000,0.578483,16,9,12,13
3,qwen25_coder_7b,CASTLE_all_250,C_errorlock_same_dataset,250,250,1.000,0.632000,0.636218,0.641667,0.629630,0.657308,0.632000,0.635556,69,31,61,89
4,qwen25_coder_7b,Juliet_overlap_250,E_baseline,250,249,0.996,0.686747,0.766036,0.687839,0.662297,0.766624,0.686747,0.661932,52,73,5,119
5,qwen25_coder_7b,Juliet_overlap_250,F_errorlock_from_castle,250,250,1.000,0.824000,0.841116,0.824000,0.821764,0.841116,0.824000,0.821764,89,36,8,117


,model,dataset,method,total,parsed,parsed_rate,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,tn,fp,fn,tp
0,qwen25_coder_7b,CASTLE_all_250,A_baseline,250,237,0.948,0.649789,0.624871,0.608508,0.609648,0.638211,0.649789,0.637641,39,53,30,115
1,qwen25_coder_7b,CASTLE_heldout_50,B0_baseline_on_heldout,50,47,0.940,0.680851,0.686275,0.672727,0.671329,0.685023,0.680851,0.674900,12,10,5,20
2,qwen25_coder_7b,CASTLE_heldout_50,B_errorlock_train_split,50,50,1.000,0.580000,0.581169,0.580000,0.578483,0.581169,0.580000,0.578483,16,9,12,13
3,qwen25_coder_7b,CASTLE_all_250,C_errorlock_same_dataset,250,250,1.000,0.632000,0.636218,0.641667,0.629630,0.657308,0.632000,0.635556,69,31,61,89
4,qwen25_coder_7b,Juliet_overlap_250,E_baseline,250,249,0.996,0.686747,0.766036,0.687839,0.662297,0.766624,0.686747,0.661932,52,73,5,119
5,qwen25_coder_7b,Juliet_overlap_250,F_errorlock_from_castle,250,250,1.000,0.824000,0.841116,0.824000,0.821764,0.841116,0.824000,0.821764,89,36,8,117


## 13. Reload metrics without rerunning

In [21]:
final_metrics_path = RESULT_DIR / "summary_metrics_final.csv"
partial_metrics_path = RESULT_DIR / "summary_metrics_partial.csv"

if final_metrics_path.exists():
    metrics_df = pd.read_csv(final_metrics_path)
elif partial_metrics_path.exists():
    metrics_df = pd.read_csv(partial_metrics_path)
else:
    raise FileNotFoundError("No metrics file found yet. Run the experiment loop first.")

metrics_df

,model,dataset,method,total,parsed,parsed_rate,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,tn,fp,fn,tp
0,qwen25_coder_7b,CASTLE_all_250,A_baseline,250,237,0.948,0.649789,0.624871,0.608508,0.609648,0.638211,0.649789,0.637641,39,53,30,115
1,qwen25_coder_7b,CASTLE_heldout_50,B0_baseline_on_heldout,50,47,0.940,0.680851,0.686275,0.672727,0.671329,0.685023,0.680851,0.674900,12,10,5,20
2,qwen25_coder_7b,CASTLE_heldout_50,B_errorlock_train_split,50,50,1.000,0.580000,0.581169,0.580000,0.578483,0.581169,0.580000,0.578483,16,9,12,13
3,qwen25_coder_7b,CASTLE_all_250,C_errorlock_same_dataset,250,250,1.000,0.632000,0.636218,0.641667,0.629630,0.657308,0.632000,0.635556,69,31,61,89
4,qwen25_coder_7b,Juliet_overlap_250,E_baseline,250,249,0.996,0.686747,0.766036,0.687839,0.662297,0.766624,0.686747,0.661932,52,73,5,119
5,qwen25_coder_7b,Juliet_overlap_250,F_errorlock_from_castle,250,250,1.000,0.824000,0.841116,0.824000,0.821764,0.841116,0.824000,0.821764,89,36,8,117


## 14. Report-ready metrics table

In [22]:
display_cols = [
    "model", "dataset", "method", "total", "parsed", "parsed_rate",
    "accuracy", "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted",
    "tn", "fp", "fn", "tp",
]

report_table = metrics_df[display_cols].copy()

for col in [
    "parsed_rate", "accuracy", "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted",
]:
    report_table[col] = report_table[col].apply(lambda x: round(x, 4) if pd.notna(x) else x)

report_table.to_csv(RESULT_DIR / "report_table.csv", index=False)
report_table

,model,dataset,method,total,parsed,parsed_rate,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,tn,fp,fn,tp
0,qwen25_coder_7b,CASTLE_all_250,A_baseline,250,237,0.948,0.6498,0.6249,0.6085,0.6096,0.6382,0.6498,0.6376,39,53,30,115
1,qwen25_coder_7b,CASTLE_heldout_50,B0_baseline_on_heldout,50,47,0.940,0.6809,0.6863,0.6727,0.6713,0.6850,0.6809,0.6749,12,10,5,20
2,qwen25_coder_7b,CASTLE_heldout_50,B_errorlock_train_split,50,50,1.000,0.5800,0.5812,0.5800,0.5785,0.5812,0.5800,0.5785,16,9,12,13
3,qwen25_coder_7b,CASTLE_all_250,C_errorlock_same_dataset,250,250,1.000,0.6320,0.6362,0.6417,0.6296,0.6573,0.6320,0.6356,69,31,61,89
4,qwen25_coder_7b,Juliet_overlap_250,E_baseline,250,249,0.996,0.6867,0.7660,0.6878,0.6623,0.7666,0.6867,0.6619,52,73,5,119
5,qwen25_coder_7b,Juliet_overlap_250,F_errorlock_from_castle,250,250,1.000,0.8240,0.8411,0.8240,0.8218,0.8411,0.8240,0.8218,89,36,8,117


## 15. Baseline vs ErrorLock comparison table

In [23]:
def compare_methods(metrics_df, model_name, dataset_name, baseline_method, errorlock_method):
    base = metrics_df[
        (metrics_df["model"] == model_name) &
        (metrics_df["dataset"] == dataset_name) &
        (metrics_df["method"] == baseline_method)
    ]

    err = metrics_df[
        (metrics_df["model"] == model_name) &
        (metrics_df["dataset"] == dataset_name) &
        (metrics_df["method"] == errorlock_method)
    ]

    if len(base) == 0 or len(err) == 0:
        return None

    base = base.iloc[0]
    err = err.iloc[0]

    rows = []

    for metric in ["accuracy", "f1_macro", "f1_weighted", "fp", "fn", "parsed_rate"]:
        rows.append({
            "model": model_name,
            "dataset": dataset_name,
            "metric": metric,
            "baseline": base[metric],
            "errorlock": err[metric],
            "change": err[metric] - base[metric],
        })

    return pd.DataFrame(rows)


comparisons = []

for model_short in sorted(metrics_df["model"].unique()):
    comp_b = compare_methods(metrics_df, model_short, "CASTLE_heldout_50", "B0_baseline_on_heldout", "B_errorlock_train_split")
    if comp_b is not None:
        comparisons.append(comp_b)

    comp_c = compare_methods(metrics_df, model_short, "CASTLE_all_250", "A_baseline", "C_errorlock_same_dataset")
    if comp_c is not None:
        comparisons.append(comp_c)

    comp_f = compare_methods(metrics_df, model_short, "Juliet_overlap_250", "E_baseline", "F_errorlock_from_castle")
    if comp_f is not None:
        comparisons.append(comp_f)

if len(comparisons) > 0:
    comparison_table = pd.concat(comparisons, ignore_index=True)
    comparison_table.to_csv(RESULT_DIR / "baseline_vs_errorlock_changes.csv", index=False)
    display(comparison_table)
else:
    print("No comparisons found yet.")

,model,dataset,metric,baseline,errorlock,change
0,qwen25_coder_7b,CASTLE_heldout_50,accuracy,0.680851,0.580000,-0.100851
1,qwen25_coder_7b,CASTLE_heldout_50,f1_macro,0.671329,0.578483,-0.092846
2,qwen25_coder_7b,CASTLE_heldout_50,f1_weighted,0.674900,0.578483,-0.096417
3,qwen25_coder_7b,CASTLE_heldout_50,fp,10.000000,9.000000,-1.000000
4,qwen25_coder_7b,CASTLE_heldout_50,fn,5.000000,12.000000,7.000000
5,qwen25_coder_7b,CASTLE_heldout_50,parsed_rate,0.940000,1.000000,0.060000
6,qwen25_coder_7b,CASTLE_all_250,accuracy,0.649789,0.632000,-0.017789
7,qwen25_coder_7b,CASTLE_all_250,f1_macro,0.609648,0.629630,0.019981
8,qwen25_coder_7b,CASTLE_all_250,f1_weighted,0.637641,0.635556,-0.002086
9,qwen25_coder_7b,CASTLE_all_250,fp,53.000000,31.000000,-22.000000


## 16. Inspect retrieval traces

In [24]:
def load_result_csv(pattern):
    files = sorted(RESULT_DIR.glob(pattern))
    print("Matching files:")
    for f in files:
        print(" ", f)
    if not files:
        return None
    return pd.read_csv(files[0])


# Example:
# trace_df = load_result_csv("*__F_juliet_castle_errorlock.csv")
# trace_df[["id", "true_label", "pred_label", "retrieved_mistake_ids", "retrieved_error_types", "retrieved_scores"]].head()